# 07 - Downstream Defect Evaluation
**Addresses comment 4 (manufacturing relevance).**  PSNR/SSIM do not show that a defect is still visible after restoration. Protocol:

1. Train **one** defect segmenter on **clean** Carinthia-S images (train part of a seeded 70/30 split of the verified images that have masks). It is frozen and shared by every arm.
2. Degrade the held-out clean test images with the **estimated** degradation (notebook 00), restore with every model, and run the *same* segmenter.
3. Report, against ground-truth defect masks: pixel precision/recall/F1/IoU; **defect-level recall (miss rate), precision and false alarms per image**; plus defect **contrast-to-noise**, **edge correlation**, flat-region artefact error and full-scale **HP-Err**.
4. Arms: `clean` (upper bound), `bilinear` (lower bound), baseline, FiLM-control, full, external models. Paired image-level tests with Holm correction.

**Caveats to state:** restoration models are trained on KLA data and applied to Carinthia-S (domain shift - optionally fine-tune on Carinthia-S train images); the segmenter is small; defects are connected components, with a minimum area and a 10% overlap rule.

In [8]:
import os, sys, json, warnings, importlib
PROJECT_ROOT = os.path.abspath("..")
if not os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
# Prefer a configured data root only when it actually contains the paired folders.
def _has_paired_data(root):
    if not root:
        return False
    return ((os.path.isdir(os.path.join(root, "train", "train", "GT")) and
             os.path.isdir(os.path.join(root, "train", "train", "NoisyLR"))) or
            (os.path.isdir(os.path.join(root, "GT")) and
             os.path.isdir(os.path.join(root, "NoisyLR"))))
if not _has_paired_data(os.environ.get("SEMICON_DATA")):
    os.environ["SEMICON_DATA"] = PROJECT_ROOT
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C = importlib.reload(C)
# Print explicit results and data roots below
from utils import carinthia as CA, downstream as DS, stats as S, figures as FG, metrics as MT
from utils.degrade import degrade, fit_degradation
from utils.analysis import finished_seeds, load_run_model, restore
RUNS = os.path.abspath(os.environ.get("SEMICON_RUNS", os.path.join(PROJECT_ROOT, "results_kaggle")))
DATA = os.path.abspath(os.environ.get("SEMICON_DATA", PROJECT_ROOT))
C.OUT = RUNS
C.RUNS = os.path.join(RUNS, "runs")
C.TABLES = os.path.join(RUNS, "tables")
C.FIGS = os.path.join(RUNS, "figures")
C.DATA_ROOT = DATA
print("RUNS =", RUNS)
print("DATA =", DATA)
if not os.path.isfile(os.path.join(RUNS, "split_manifest.json")):
    raise FileNotFoundError(f"Kaggle results split manifest not found under RUNS={RUNS}")
splits, man = C.get_splits()
fit_path = os.path.join(C.TABLES, "degradation_fit.json")
fit = json.load(open(fit_path)) if os.path.exists(fit_path) else fit_degradation(splits["train"])
LOOKS, SIGMA = fit["looks"], fit["sigma"]
dev = C.device()
CARINTHIA_DATA = os.path.abspath(os.environ.get("SEMICON_CARINTHIA", os.path.join(DATA, "data", "data", "data")))
C.CARINTHIA_ROOT = CARINTHIA_DATA
print("CARINTHIA_DATA =", CARINTHIA_DATA)
root = CARINTHIA_DATA
if C.SMOKE:
    root = os.path.join(C.OUT, "synthetic_carinthia")
    CA.make_synthetic_carinthia(root, n=24, size=64)
RUN_DOWNSTREAM = os.path.isdir(root)
if not RUN_DOWNSTREAM:
    print("Carinthia-S images and masks were not found at", root)
    print("Downstream defect evaluation is skipped. Extract Carinthia-S and set SEMICON_CARINTHIA to its dataset root to run it.")
    items, tr_i, te_i = [], [], []
else:
    SIZE = 64 if C.SMOKE else 256
    items = CA.load_pairs(root, n=200 if C.MODE == "full" else 60, seed=0, size=SIZE, require_masks=True,
                          manifest_path=os.path.join(C.TABLES, "carinthia_downstream_manifest.json"))
    rng = np.random.RandomState(0)
    order = rng.permutation(len(items)); ntr = int(0.7 * len(items))
    tr_i, te_i = [items[i] for i in order[:ntr]], [items[i] for i in order[ntr:]]
    print(f"{len(tr_i)} train / {len(te_i)} test images with masks; defect pixel rate {np.mean([i['mask'].mean() for i in items]):.4f}")
    T = lambda lst, k: torch.from_numpy(np.stack([i[k] for i in lst]).astype(np.float32))[:, None]

RUNS = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle
DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow
CARINTHIA_DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\data\data\data
42 train / 18 test images with masks; defect pixel rate 0.0214


In [9]:
if RUN_DOWNSTREAM:
    seg = DS.train_segmenter(T(tr_i, "image"), T(tr_i, "mask"), epochs={"smoke": 3, "pilot": 25, "full": 60}[C.MODE], seed=0, device=dev)
    clean = T(te_i, "image"); masks = T(te_i, "mask")
    up = lambda a: torch.from_numpy(np.kron(a, np.ones((2, 2), np.float32)))[None, None]
    names = [c for c in ["baseline", "film_control", "full", "NAFNet-small", "Restormer-small", "SwinIR-small"] if finished_seeds(C.RUNS, c)]
    seeds_by = {c: finished_seeds(C.RUNS, c) for c in names}; print(seeds_by)
else:
    names, seeds_by = [], {}
    print("Segmenter training skipped because Carinthia-S is unavailable.")

{'baseline': [0, 1, 2, 3], 'film_control': [0, 1, 2, 3], 'full': [0, 1, 2, 3]}


In [10]:
if RUN_DOWNSTREAM:
    arms = {}
    noise_draws = (0,) if C.MODE == "smoke" else (0, 1, 2)
    def rows_to_df(rows, arm, seed, nd):
        d = pd.DataFrame(rows); d["arm"], d["seed"], d["noise"], d["img"] = arm, seed, nd, np.arange(len(d)); return d
    all_rows = [rows_to_df(DS.evaluate_arm(seg, clean, clean, clean[:, :, ::2, ::2], masks, dev), "clean (upper bound)", 0, 0)]
    for nd in noise_draws:
        deg = torch.from_numpy(np.stack([degrade(clean[k, 0].numpy(), LOOKS, SIGMA, seed=100 * nd + k) for k in range(len(clean))]))[:, None]
        all_rows.append(rows_to_df(DS.evaluate_arm(seg, torch.cat([up(deg[k, 0].numpy()) for k in range(len(deg))]).clamp(0, 1), clean, deg, masks, dev), "bilinear (lower bound)", 0, nd))
        for c in names:
            for sd in seeds_by[c]:
                model, cfg, _ = load_run_model(C.RUNS, c, sd, dev)
                with torch.no_grad(): pred = restore(model, cfg, deg.to(dev)).clamp(0, 1).cpu()
                all_rows.append(rows_to_df(DS.evaluate_arm(seg, pred, clean, deg, masks, dev), c, sd, nd))
    D = pd.concat(all_rows, ignore_index=True); D.to_csv(os.path.join(C.TABLES, "downstream_long.csv"), index=False)
    print(D.shape, "rows")
else:
    print("Downstream metric computation skipped because Carinthia-S is unavailable.")

(720, 21) rows


In [11]:
if RUN_DOWNSTREAM:
    cols = ["pix_f1", "inst_recall", "inst_precision", "false_alarms_per_img", "defect_cnr_ratio_to_clean", "edge_corr", "hp_err", "flat_err", "psnr"]
    per = D.groupby(["arm", "img"])[cols].mean().reset_index()
    rows = []
    for a, g in per.groupby("arm"):
        r = {"arm": a, "n_test_images": len(g)}
        for c in cols:
            v = g[c].dropna().values; lo, hi = S.bootstrap_ci(v, 2000) if len(v) > 1 else (np.nan, np.nan)
            r[c] = f"{v.mean():.3f} [{lo:.3f},{hi:.3f}]"
        rows.append(r)
    tabD = pd.DataFrame(rows).set_index("arm").loc[[a for a in ["clean (upper bound)", "bilinear (lower bound)"] + names if a in per.arm.unique()]]
    print(tabD.T.to_string()); tabD.to_csv(os.path.join(C.TABLES, "table_downstream.csv"))
else:
    print("Downstream summary skipped because Carinthia-S is unavailable.")

arm                              clean (upper bound)  bilinear (lower bound)                baseline            film_control                    full
n_test_images                                     18                      18                      18                      18                      18
pix_f1                           0.425 [0.301,0.535]     0.329 [0.224,0.432]     0.387 [0.270,0.493]     0.396 [0.278,0.502]     0.393 [0.275,0.500]
inst_recall                      1.000 [1.000,1.000]     1.000 [1.000,1.000]     0.990 [0.969,1.000]     0.984 [0.953,1.000]     0.987 [0.961,1.000]
inst_precision                   0.528 [0.398,0.667]     0.124 [0.092,0.154]     0.300 [0.214,0.391]     0.333 [0.228,0.452]     0.334 [0.234,0.448]
false_alarms_per_img             1.167 [0.722,1.612]   10.852 [7.166,15.427]     4.032 [2.690,5.621]     3.731 [2.444,5.316]     3.505 [2.319,4.926]
defect_cnr_ratio_to_clean        1.000 [1.000,1.000]     0.810 [0.755,0.854]     0.981 [0.937,1.016]     0

In [12]:
if RUN_DOWNSTREAM:
    out = []
    for m, hb in [("pix_f1", True), ("inst_recall", True), ("inst_precision", True), ("false_alarms_per_img", False), ("defect_cnr_ratio_to_clean", True), ("edge_corr", True)]:
        P = {a: per[per.arm == a].sort_values("img")[m].fillna(0).values for a in per.arm.unique()}
        for ref in [r for r in ["bilinear (lower bound)", "baseline"] if r in P]:
            t = S.compare_to_reference(P, ref, [a for a in P if a != "clean (upper bound)"], higher_better=hb); t["metric"] = m; out.append(t)
    tt = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    if not tt.empty:
        tt["p_holm_all"] = S.holm(tt.p_wilcoxon.values)
        print(tt[["metric", "config", "vs", "mean_diff", "ci95_lo", "ci95_hi", "p_holm_all", "cohens_dz"]].round(4).to_string(index=False))
        tt.to_csv(os.path.join(C.TABLES, "table_downstream_paired.csv"), index=False)
        if "full" in names and "baseline" in names:
            for m in ["inst_recall", "false_alarms_per_img", "pix_f1"]:
                r = tt[(tt.metric == m) & (tt.config == "full") & (tt.vs == "baseline")]
                if len(r):
                    r = r.iloc[0]
                    print(f"downstream claim 'full beats baseline' on {m}: {'SUPPORTED' if (r.mean_diff>0 and r.ci95_lo>0 and r.p_holm_all<0.05) else 'NOT SUPPORTED'} (diff {r.mean_diff:+.4f}, CI [{r.ci95_lo:+.4f},{r.ci95_hi:+.4f}])")
    else:
        print("No paired comparisons are available.")
else:
    print("Downstream paired tests skipped because Carinthia-S is unavailable.")

                   metric                 config                     vs  mean_diff  ci95_lo  ci95_hi  p_holm_all  cohens_dz
                   pix_f1               baseline bilinear (lower bound)     0.0580   0.0422   0.0734      0.0114     1.6511
                   pix_f1           film_control bilinear (lower bound)     0.0673   0.0485   0.0861      0.0114     1.5876
                   pix_f1                   full bilinear (lower bound)     0.0642   0.0466   0.0815      0.0114     1.6317
                   pix_f1 bilinear (lower bound)               baseline    -0.0580  -0.0734  -0.0422      0.0114    -1.6511
                   pix_f1           film_control               baseline     0.0093   0.0053   0.0135      0.0230     1.0279
                   pix_f1                   full               baseline     0.0062   0.0040   0.0085      0.0114     1.2538
              inst_recall               baseline bilinear (lower bound)    -0.0093  -0.0278   0.0000      1.0000    -0.2357
        

In [13]:
if RUN_DOWNSTREAM:
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.4)); arms_o = list(tabD.index)
    for a_, (m, title) in zip(ax, [("inst_recall", "Defect recall"), ("inst_precision", "Defect precision"), ("false_alarms_per_img", "False alarms / image")]):
        v = [per[per.arm == a][m].mean() for a in arms_o]; e = [S.bootstrap_ci(per[per.arm == a][m].dropna().values, 1000) for a in arms_o]
        a_.bar(range(len(arms_o)), v, yerr=[[x - l for x, (l, h) in zip(v, e)], [h - x for x, (l, h) in zip(v, e)]], capsize=3); a_.set_title(title)
        a_.set_xticks(range(len(arms_o))); a_.set_xticklabels([x.split(" (")[0] for x in arms_o], rotation=35, ha="right")
    FG.save(fig, C.FIGS, "fig_downstream", {"dataset": "Carinthia-S", "n_test": len(te_i), "looks": LOOKS, "sigma": SIGMA, "error_bars": "95% bootstrap CI over images"})
else:
    print("Downstream figure skipped because Carinthia-S is unavailable.")